# Rule-Based and ML Trading Signals on the S&P 500

**Structure:**
1. Data setup
2. Baseline: simple rule-based strategies (moving-average crossover, mean reversion)
3. Feature engineering for a machine-learning approach
4. ML pipeline with walk-forward (time-series) cross-validation
5. From predictions to profit — including transaction costs
6. Conclusions and lessons learned


## 1. Setup

In [ ]:
# pip install pandas numpy matplotlib seaborn yfinance pandas_market_calendars scikit-learn xgboost statsmodels
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set()
plt.rcParams['figure.figsize'] = 12, 6

import yfinance as yf
import pandas_market_calendars as mcal


In [ ]:
# Using the S&P 500 as the test index. Long history, highly liquid, well-arbitraged —
# i.e. a fair, hard test of whether a simple signal has any edge.
ticker = yf.Ticker("^GSPC")
raw = ticker.history(period="max")
raw = raw.drop(columns=[c for c in ["Dividends", "Stock Splits", "Volume"] if c in raw.columns])
raw.index = raw.index.tz_localize(None)
raw.tail()


## 2. Baseline: rule-based strategies

Before reaching for machine learning, it's worth checking what simple, well-known technical
rules achieve on their own. Two classics: a moving-average **crossover** (trend-following) and
a **mean-reversion** rule based on how far price has drifted from its recent average.

Both are backtested as long/short signals against log returns, and compared to buy-and-hold on
the same period. No transaction costs are included yet — that's added later once we get to a
strategy we'd seriously consider.

In [ ]:
df_rb = raw.copy()

df_rb['50-day'] = df_rb['Close'].rolling(50).mean().shift()
df_rb['200-day'] = df_rb['Close'].rolling(200).mean().shift()

# Long when the short-term average is above the long-term average, short otherwise
df_rb['signal'] = np.where(df_rb['50-day'] > df_rb['200-day'], 1, -1)
df_rb.dropna(inplace=True)

df_rb['return'] = np.log(df_rb['Close']).diff()
df_rb['system_return'] = df_rb['return'] * df_rb['signal']
df_rb['entry'] = df_rb['signal'].diff()


In [ ]:
# Visualise the last ~4 years of crossovers
window = df_rb.iloc[-1052:]
plt.plot(window['Close'], label='Close')
plt.plot(window['50-day'], label='50-day MA')
plt.plot(window['200-day'], label='200-day MA')
plt.plot(window.loc[window.entry == 2].index, window['50-day'][window.entry == 2], '^', color='g', markersize=10, label='Long entry')
plt.plot(window.loc[window.entry == -2].index, window['200-day'][window.entry == -2], 'v', color='r', markersize=10, label='Short entry')
plt.legend(loc='upper left')
plt.title('50/200-day Moving Average Crossover')
plt.show()


In [ ]:
plt.plot(np.exp(window['return']).cumprod(), label='Buy & Hold')
plt.plot(np.exp(window['system_return']).cumprod(), label='Crossover Strategy')
plt.legend(loc='upper left')
plt.title('Crossover strategy vs. Buy & Hold (no costs)')
plt.show()

print(f"Full-history log return — Buy & Hold: {df_rb['return'].sum():.3f}, Crossover: {df_rb['system_return'].sum():.3f}")


### 2.2 Mean reversion

The idea: when price is unusually far above its recent moving average, bet on it falling back;
when unusually far below, bet on it rising. Bands are set from the historical 5th/95th percentile
of the price-to-moving-average ratio, rather than an arbitrary fixed threshold.

In [ ]:
df_mr = raw.copy()
ma_window = 21

df_mr['returns'] = np.log(df_mr['Close']).diff()
df_mr['ma'] = df_mr['Close'].rolling(ma_window).mean()
df_mr['ratio'] = df_mr['Close'] / df_mr['ma']

lower_pct, upper_pct = np.percentile(df_mr['ratio'].dropna(), [5, 95])

df_mr['position'] = np.where(df_mr['ratio'] > upper_pct, -1, np.nan)
df_mr['position'] = np.where(df_mr['ratio'] < lower_pct, 1, df_mr['position'])
df_mr['position'] = df_mr['position'].ffill()

df_mr['strat_return'] = df_mr['returns'] * df_mr['position'].shift()


In [ ]:
window_mr = df_mr.iloc[-1052:]
plt.plot(np.exp(window_mr['returns'].dropna()).cumprod(), label='Buy & Hold')
plt.plot(np.exp(window_mr['strat_return'].dropna()).cumprod(), label='Mean Reversion Strategy')
plt.legend(loc='upper left')
plt.title('Mean reversion strategy vs. Buy & Hold (no costs)')
plt.show()


**Baseline takeaway:** both rules occasionally beat buy-and-hold over specific windows, but
neither shows a consistent, robust edge across the full history once you account for the fact
that these parameters (50/200 day, 21-day, 5th/95th percentile) were chosen with hindsight over
the same data being tested.

## 3. Feature engineering for a machine-learning approach

Rather than hand-picking one or two signals, this section builds a broader feature set —
standard technical indicators plus calendar effects — that a model can weigh automatically.

In [ ]:
def add_technical_indicators(df):
    """
    Given a DataFrame of OHLC data (columns: Open, High, Low, Close), return a copy
    with standard technical indicators added: moving averages, momentum, volatility,
    RSI, MACD, Bollinger Bands, and Average True Range.
    """
    df = df.copy()

    df['SMA_10'] = df['Close'].rolling(10).mean()
    df['SMA_50'] = df['Close'].rolling(50).mean()
    df['EMA_10'] = df['Close'].ewm(span=10, adjust=False).mean()
    df['EMA_50'] = df['Close'].ewm(span=50, adjust=False).mean()

    df['Momentum_10'] = df['Close'] - df['Close'].shift(10)
    df['Volatility_10'] = df['Close'].pct_change().rolling(10).std()

    delta = df['Close'].diff()
    gain = delta.clip(lower=0).rolling(14).mean()
    loss = (-delta.clip(upper=0)).rolling(14).mean()
    rs = gain / loss
    df['RSI_14'] = 100 - (100 / (1 + rs))

    ema12 = df['Close'].ewm(span=12, adjust=False).mean()
    ema26 = df['Close'].ewm(span=26, adjust=False).mean()
    df['MACD'] = ema12 - ema26
    df['MACD_Signal'] = df['MACD'].ewm(span=9, adjust=False).mean()

    df['BB_Middle'] = df['Close'].rolling(20).mean()
    df['BB_Std'] = df['Close'].rolling(20).std()
    df['BB_Upper'] = df['BB_Middle'] + 2 * df['BB_Std']
    df['BB_Lower'] = df['BB_Middle'] - 2 * df['BB_Std']

    true_range = np.maximum(
        df['High'] - df['Low'],
        np.maximum((df['High'] - df['Close'].shift(1)).abs(),
                   (df['Low'] - df['Close'].shift(1)).abs())
    )
    df['ATR_14'] = true_range.rolling(14).mean()

    return df


def add_calendar_features(df):
    """
    Add weekday and day-of-year (cyclically encoded) features. Cyclical encoding
    avoids treating e.g. day 365 and day 1 as far apart, which a raw integer would imply.
    """
    df = df.copy()
    df['weekday'] = df.index.weekday
    df['dayofyear'] = df.index.dayofyear
    df['dofy_sin'] = np.sin(2 * np.pi * df['dayofyear'] / 365.25)
    df['dofy_cos'] = np.cos(2 * np.pi * df['dayofyear'] / 365.25)
    return df


In [ ]:
ml_df = raw.copy()
ml_df = add_technical_indicators(ml_df)
ml_df = add_calendar_features(ml_df)
ml_df.tail()


**Label:** whether the *next* day's close is higher than today's close. This is a simple
binary classification target, deliberate since the point of this project was to test
the validation methodology.

In [ ]:
ml_df['fwd_return'] = ml_df['Close'].pct_change().shift(-1)
ml_df['Signal'] = np.where(ml_df['fwd_return'] > 0, 1, 0)   # 1 = Up, 0 = Down

ml_df = ml_df.dropna().copy()   # drops rows with NaNs from rolling windows and the final row (no forward return)
ml_df['Signal'].value_counts(normalize=True)


## 4. ML pipeline with walk-forward validation

The critical design choice here is **`TimeSeriesSplit`** rather than a random train/test split.
Shuffling time-series data before splitting would let the model "see the future" relative to
some of its training points, a classic and easy-to-miss source of inflated backtest results.
Walk-forward validation trains only on data *before* each test window.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                              confusion_matrix, ConfusionMatrixDisplay)

feature_cols = ['SMA_10', 'SMA_50', 'EMA_10', 'EMA_50', 'Momentum_10', 'Volatility_10',
                'RSI_14', 'MACD', 'MACD_Signal', 'ATR_14',
                'weekday', 'dofy_sin', 'dofy_cos']

X = ml_df[feature_cols]
y = ml_df['Signal'].values

preprocessor = ColumnTransformer([
    ('impute_scale', Pipeline([
        ('impute', SimpleImputer(strategy='median')),
        ('scale', StandardScaler())
    ]), feature_cols)
])

models = {
    "LogReg": LogisticRegression(max_iter=1000),
    "RandomForest": RandomForestClassifier(n_estimators=200, random_state=42),
    "GradientBoost": GradientBoostingClassifier(),
    "KNN": KNeighborsClassifier(),
    "XGBoost": XGBClassifier(eval_metric='logloss'),
    "SVC": SVC(probability=True),
}


In [ ]:
def walk_forward_eval(model, X, y, n_splits=8, test_size=100):
    """
    Runs walk-forward (expanding-window) time-series cross-validation for a given
    sklearn-compatible model. Returns concatenated out-of-sample predictions and truths,
    plus per-fold accuracy for a quick sanity check of stability across folds.
    """
    tscv = TimeSeriesSplit(n_splits=n_splits, test_size=test_size)
    all_preds, all_truth, fold_accs = [], [], []

    for train_idx, test_idx in tscv.split(X):
        model.fit(X.iloc[train_idx], y[train_idx])
        preds = model.predict(X.iloc[test_idx])

        all_preds.append(preds)
        all_truth.append(y[test_idx])
        fold_accs.append(accuracy_score(y[test_idx], preds))

    return np.concatenate(all_preds), np.concatenate(all_truth), fold_accs


In [ ]:
results = {}

for name, clf in models.items():
    pipe = Pipeline([('preprocessing', preprocessor), ('classifier', clf)])
    preds, truth, fold_accs = walk_forward_eval(pipe, X, y, n_splits=8, test_size=100)
    acc = accuracy_score(truth, preds)
    bal_acc = balanced_accuracy_score(truth, preds)
    results[name] = {'accuracy': acc, 'balanced_accuracy': bal_acc, 'fold_accuracy_std': np.std(fold_accs)}
    print(f"{name:14s}  accuracy={acc:.3f}  balanced_accuracy={bal_acc:.3f}  fold_std={np.std(fold_accs):.3f}")

results_df = pd.DataFrame(results).T.sort_values('balanced_accuracy', ascending=False)
results_df


None of these surpassed ~55-60% accuracy by much, if at all.  A confusion matrix for the
best-performing model below

In [ ]:
best_model_name = results_df.index[0]
best_pipe = Pipeline([('preprocessing', preprocessor), ('classifier', models[best_model_name])])
preds, truth, _ = walk_forward_eval(best_pipe, X, y, n_splits=8, test_size=100)

cm = confusion_matrix(truth, preds, labels=[0, 1])
ConfusionMatrixDisplay(cm, display_labels=['Down', 'Up']).plot()
plt.title(f'Confusion matrix — {best_model_name}')
plt.show()


## 5. From predictions to profit

Classification accuracy alone doesn't tell you whether a strategy makes money - a model can be
"right" more than half the time and still lose, if wrong calls are costlier than right ones are
profitable, or if trading costs eat the edge. This section converts predictions into an actual
position (long when predicting "up", short when predicting "down") and applies a simple
per-trade cost assumption, then compares cumulative return to buy-and-hold over the same
out-of-sample window.

In [ ]:
cost_per_trade = 0.0005  # 5 basis points round-trip, a rough placeholder for spread + fees

last_train_idx, last_test_idx = list(TimeSeriesSplit(n_splits=8, test_size=100).split(X))[-1]
best_pipe.fit(X.iloc[last_train_idx], y[last_train_idx])

test_slice = ml_df.iloc[last_test_idx].copy()
test_slice['pred'] = best_pipe.predict(X.iloc[last_test_idx])
test_slice['position'] = np.where(test_slice['pred'] == 1, 1, -1)

# Cost is charged whenever the position changes from the previous day
test_slice['position_change'] = test_slice['position'].diff().abs().fillna(1)
test_slice['strategy_return'] = (test_slice['position'] * test_slice['fwd_return']
                                  - test_slice['position_change'] * cost_per_trade)
test_slice['buyhold_return'] = test_slice['fwd_return']

cum_strategy = (1 + test_slice['strategy_return']).cumprod()
cum_buyhold = (1 + test_slice['buyhold_return']).cumprod()

plt.plot(cum_strategy.values, label=f'{best_model_name} strategy (with costs)')
plt.plot(cum_buyhold.values, label='Buy & Hold')
plt.legend()
plt.title('Out-of-sample cumulative return — final walk-forward test window')
plt.xlabel('Trading day (test window)')
plt.ylabel('Cumulative return (x)')
plt.show()

print(f"Final cumulative return — Strategy: {cum_strategy.iloc[-1]:.3f}, Buy & Hold: {cum_buyhold.iloc[-1]:.3f}")


It's also worth checking what happens **without** the cost assumption, since the gap between
the two numbers *is* the finding: it shows how much of any apparent edge is actually just being
handed back in trading costs.

In [ ]:
test_slice['strategy_return_no_cost'] = test_slice['position'] * test_slice['fwd_return']
cum_strategy_no_cost = (1 + test_slice['strategy_return_no_cost']).cumprod()
print(f"Final cumulative return, no costs assumed: {cum_strategy_no_cost.iloc[-1]:.3f}")
print(f"Final cumulative return, with 5bps round-trip cost: {cum_strategy.iloc[-1]:.3f}")
